
# Etapa 3 — Reconhecimento de Entidades Nomeadas (NER)

**Objetivo desta etapa:** implementar e avaliar um modelo de Reconhecimento de Entidades Nomeadas (*Named Entity Recognition — NER*) para identificar automaticamente marcas, produtos, organizações e pessoas mencionadas em textos em português.

**Dados:** avaliações de produtos da base RePro e exemplos de comentários em português no estilo YouTube. Posteriormente, serão utilizados comentários reais coletados do YouTube para avaliar o desempenho do modelo.

**Modelo:** GLiNER2 PT-BR ([ottema/gliner2-ptbr](https://huggingface.co/ottema/gliner2-ptbr)), modelo pré-treinado disponibilizado no Hugging Face, voltado à extração de entidades em português brasileiro.

**Roteiro do notebook:**

1. Preparar o ambiente e carregar o modelo pré-treinado.
2. Carregar os dados e selecionar exemplos para análise.
3. Definir as categorias de entidades (marcas, produtos, organizações e pessoas).
4. Aplicar o modelo NER aos textos selecionados.
5. Visualizar as entidades identificadas.
6. Testar novos comentários no estilo YouTube.
7. Avaliar o reconhecimento com exemplos anotados manualmente.
8. Registrar entidades não identificadas e classificações incorretas.
9. Analisar os resultados e as limitações do modelo.

**Integração com a Etapa 2:** o modelo NER será utilizado em conjunto com o classificador de sentimentos desenvolvido anteriormente, permitindo identificar tanto as entidades mencionadas quanto os sentimentos expressos nos comentários.


## 1. Carregar os dados

In [7]:
%pip install gliner2 torch

  Using cached gliner2-2.1.0-py3-none-any.whl.metadata (57 kB)
  Using cached pydantic-2.14.2-py3-none-any.whl.metadata (137 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metadata (2.6 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached certifi-2026.10.23-py3-none-any.whl.metadata (2.5 kB)
  Using cached filelock-4.2.7-py3-none-any.whl.metadata (2.0 kB)
  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmat

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import torch
print(torch.__version__)

In [ ]:
from gliner2 import AutoExtractor

extractor = AutoExtractor.from_pretrained("ottema/gliner2-ptbr")

text = "Apple CEO Tim Cook announced iPhone 15 in Cupertino yesterday."
result = extractor.extract_entities(text, ["company", "person", "product", "location"])

print(result)

# 2. Testar os comentarios

In [ ]:
comentarios = [
    "Comprei um Samsung Galaxy S24 e achei a câmera incrível!",
    "Meu iPhone 15 da Apple começou a travar depois da atualização.",
    "A Natura tem produtos ótimos, mas o perfume Kaiak não dura muito.",
    "Prefiro os celulares da Motorola aos da Xiaomi."
]

categorias = ["marca", "produto", "empresa", "pessoa"]

for comentario in comentarios:
    resultado = extractor.extract_entities(comentario, categorias)

    print(f"\nComentário: {comentario}")
    print(f"Entidades: {resultado}")
    print("-" * 50)

# 3. Carregar a base das americanas - Repro

In [ ]:
import os
from pathlib import Path

os.chdir(
    r"C:\Users\Thais\Documents\social-listening-yt\src-pln\notebooks"
)

print("Diretório atual:", Path.cwd())

In [ ]:
import ast, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 200)
SEED = 42

# O arquivo é baixado por `python utils/baixar_dados.py` (rodar a partir de src-pln/)
CAMINHO = Path("../data/RePro.csv")
df = pd.read_csv(CAMINHO)
print(df.shape)
df[["product_name", "product_brand", "review_title", "review_text", "overall_rating", "polarity"]].head()

# 4. Aplicar NER nos comentarios

In [ ]:
# Selecionar 10 comentários com texto
amostra = df.dropna(subset=["review_text"]).sample(
    n=10,
    random_state=SEED
)

categorias = ["marca", "produto", "empresa", "pessoa"]

# Aplicar o modelo aos comentários
for _, linha in amostra.iterrows():
    comentario = str(linha["review_text"])

    resultado = extractor.extract_entities(
        comentario,
        categorias
    )

    print("\nComentário:", comentario)
    print("Marca cadastrada:", linha["product_brand"])
    print("Produto cadastrado:", linha["product_name"])
    print("Entidades identificadas:", resultado)
    print("-" * 60)

In [ ]:
resultados_ner = []

for _, linha in amostra.iterrows():
    comentario = str(linha["review_text"])
    entidades = extractor.extract_entities(
        comentario,
        categorias
    )["entities"]

    resultados_ner.append({
        "comentario": comentario,
        "marca_cadastrada": linha["product_brand"],
        "produto_cadastrado": linha["product_name"],
        "marcas_identificadas": entidades.get("marca", []),
        "produtos_identificados": entidades.get("produto", []),
        "empresas_identificadas": entidades.get("empresa", []),
        "pessoas_identificadas": entidades.get("pessoa", [])
    })

df_ner = pd.DataFrame(resultados_ner)

display(df_ner)

# 5. 

In [ ]:
# Categorias mais específicas
categorias_novas = [
    "marca comercial",
    "nome de produto",
    "nome de empresa",
    "nome de pessoa"
]

# Reutilizar os mesmos 10 comentários
for _, linha in amostra.iterrows():
    comentario = str(linha["review_text"])

    resultado = extractor.extract_entities(
        comentario,
        categorias_novas
    )

    print("\nComentário:", comentario)
    print("Entidades:", resultado["entities"])
    print("-" * 50)
  

In [ ]:
resultados = []

for _, linha in amostra.iterrows():
    comentario = str(linha["review_text"])

    entidades = extractor.extract_entities(
        comentario,
        categorias_novas
    )["entities"]

    resultados.append({
        "comentario": comentario,
        "marca_cadastrada": linha["product_brand"],
        "produto_cadastrado": linha["product_name"],
        "marcas": entidades.get("marca comercial", []),
        "produtos": entidades.get("nome de produto", []),
        "empresas": entidades.get("nome de empresa", []),
        "pessoas": entidades.get("nome de pessoa", [])
    })

df_ner = pd.DataFrame(resultados)

display(df_ner)

# 6. testes em comentarios no estilo do YouTube

In [ ]:
comentarios_youtube = [
    "Mano, esse Galaxy S24 da Samsung tá absurdo de bom!",
    "Depois dessa atualização meu iPhone 15 ficou horrível, Apple vacilou.",
    "Alguém sabe se o Redmi Note 13 da Xiaomi vale a pena?",
    "A câmera da Motorola tá muito melhor que a da Samsung.",
    "Comprei um perfume Kaiak da Natura e gostei demais!",
    "Esse vídeo do Felipe Neto falando da Netflix ficou muito bom."
]

resultados_youtube = []

for comentario in comentarios_youtube:
    entidades = extractor.extract_entities(
        comentario,
        categorias_novas
    )["entities"]

    resultados_youtube.append({
        "comentario": comentario,
        "marcas": entidades.get("marca comercial", []),
        "produtos": entidades.get("nome de produto", []),
        "empresas": entidades.get("nome de empresa", []),
        "pessoas": entidades.get("nome de pessoa", [])
    })

df_youtube = pd.DataFrame(resultados_youtube)

display(df_youtube)


## 7. Avaliação qualitativa do modelo NER

### Metodologia

Foi utilizado o modelo GLiNER2 PT-BR (`ottema/gliner2-ptbr`) para identificar marcas comerciais, nomes de produtos, empresas e pessoas.

A avaliação foi realizada com uma amostra de 10 avaliações da base RePro e seis comentários simulados no estilo YouTube.

Foram analisados os acertos, os falsos positivos, as classificações incorretas e as entidades não reconhecidas.

### Resultados observados

**Avaliações RePro:**
- O modelo identificou categorias genéricas, como "tapete" e "patins".
- Foram observados falsos positivos, como "borracha" classificada como marca e "filha" como pessoa.
- Muitos comentários não mencionavam explicitamente marcas ou nomes comerciais, limitando a avaliação dessas categorias.

**Comentários simulados do YouTube:**
- Reconhecimento correto de produtos como "Galaxy S24" e "Redmi Note 13".
- Reconhecimento correto da pessoa "Felipe Neto".
- Identificação de marcas como Samsung, Xiaomi, Motorola e Apple.
- Falsos positivos, como "Mano" classificado como pessoa.
- Sobreposição de categorias, como Apple classificada também como produto.

### Limitações

O modelo apresentou dificuldade em distinguir nomes próprios de palavras comuns e em separar marcas de produtos.

A base RePro não possui anotações específicas de NER, portanto os resultados foram avaliados qualitativamente, sem cálculo de precisão, recall ou F1 para reconhecimento de entidades.

### Conclusão

Os testes indicam que o GLiNER2 é capaz de reconhecer diversas entidades comerciais em português, especialmente quando seus nomes aparecem explicitamente nos comentários.

Entretanto, o desempenho observado não é suficiente para garantir extrações confiáveis sem validação adicional.

Como melhoria futura, recomenda-se comparar o GLiNER2 com outros modelos de NER e avaliar os resultados em uma amostra anotada manualmente.
